<a href="https://colab.research.google.com/github/IA-MachineLearning-Relatorios/Regression-Trees-Pruning/blob/main/Regression%26Prunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-02-car-price/data.csv")

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11914 entries, 0 to 11913
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Make               11914 non-null  object 
 1   Model              11914 non-null  object 
 2   Year               11914 non-null  int64  
 3   Engine Fuel Type   11911 non-null  object 
 4   Engine HP          11845 non-null  float64
 5   Engine Cylinders   11884 non-null  float64
 6   Transmission Type  11914 non-null  object 
 7   Driven_Wheels      11914 non-null  object 
 8   Number of Doors    11908 non-null  float64
 9   Market Category    8172 non-null   object 
 10  Vehicle Size       11914 non-null  object 
 11  Vehicle Style      11914 non-null  object 
 12  highway MPG        11914 non-null  int64  
 13  city mpg           11914 non-null  int64  
 14  Popularity         11914 non-null  int64  
 15  MSRP               11914 non-null  int64  
dtypes: float64(3), int64(5

Encontradas 8 colunas obejct que rpecisam ser trnaformadas em numeros

porem foi encontrado varias colunas com valores nao nulos que teremso q aplicar a regra de imputacao de dados, aplicar mediana nesses dados faltantes para nao termos grande parte do dataset apagado

In [3]:
mediana = df['Engine HP'].median()
df['Engine HP'] = df['Engine HP'].fillna(mediana)

#teste
df['Engine HP'].isnull().sum()

np.int64(0)

muitas colunas tem valores faltnates, algumas numericas e outras categoricas, agora realzei o parciamento do dataset para resolver isto

In [5]:
X = df.drop('MSRP', axis=1)
y = df['MSRP']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [7]:
mediana_hp = X_train['Engine HP'].median()  # calculado SÓ no treino

X_train['Engine HP'] = X_train['Engine HP'].fillna(mediana_hp)
X_test['Engine HP'] = X_test['Engine HP'].fillna(mediana_hp)  # aplica o MESMO valor

mediana_Cylinders = X_train['Engine Cylinders'].median()

X_train['Engine Cylinders'] = X_train['Engine Cylinders'].fillna(mediana_Cylinders)
X_test['Engine Cylinders'] = X_test['Engine Cylinders'].fillna(mediana_Cylinders)


mediana_Doors = X_train['Number of Doors'].median()

X_train['Number of Doors'] = X_train['Number of Doors'].fillna(mediana_Doors)
X_test['Number of Doors'] = X_test['Number of Doors'].fillna(mediana_Doors)


isolando a mediana no x_train e dps aplicar o mesmo valor no x_teste ajuda a blindar o data leakage


dados categoricos, tratados com moda para um com poucos dados faltantes

e usando uma nova coluna chamada 'uknown' pois existem mais de 3mil faltantes
quanto maior o percentual de missing, maior o risco de a moda distorcer a distribuicao

In [9]:
moda_fuel = X_train['Engine Fuel Type'].mode()[0]
X_train['Engine Fuel Type'] = X_train['Engine Fuel Type'].fillna(moda_fuel)
X_test['Engine Fuel Type'] = X_test['Engine Fuel Type'].fillna(moda_fuel)


X_train['Market Category'] = X_train['Market Category'].fillna('Unknown')
X_test['Market Category'] = X_test['Market Category'].fillna('Unknown')

In [10]:
media_treino = y_train.mean()
print(f"Previsão baseline (média do MSRP no treino): {media_treino:.2f}")

Previsão baseline (média do MSRP no treino): 40807.72


In [15]:
# 1. Error
erros = y_test - media_treino

# 2. Squared
erros_quadrados = erros ** 2

# 3. Mean
mse = erros_quadrados.mean()

# 4. Root
rmse = mse ** 0.5

print(f"RMSE (calculado manualmente): {rmse:.2f}")

RMSE (calculado manualmente): 59188.58


teste de mediana para saber se o rmse vai mudar

In [13]:
#mediana_treino = y_train.median()
#y_pred_baseline_mediana = np.full(shape=y_test.shape, fill_value=mediana_treino)
#rmse_baseline_mediana = np.sqrt(mean_squared_error(y_test, y_pred_baseline_mediana))
#print(f"RMSE do Baseline (mediana): {rmse_baseline_mediana:.2f}")

RMSE do Baseline (mediana): 60040.43


convertendo colunas obejct para colunas numericas com one hot enconding

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

colunas_categoricas = X_train.select_dtypes(include='object').columns.tolist()
colunas_numericas = X_train.select_dtypes(exclude='object').columns.tolist()

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), colunas_categoricas)
    ],
    remainder='passthrough'  # mantém as colunas numéricas sem alterar
)

X_train_final = preprocessador.fit_transform(X_train)
X_test_final = preprocessador.transform(X_test)

print(X_train_final.shape)

(8339, 1055)
